# Wheat Leaf Disease Detection — Training Notebook (Google Colab)

This notebook trains the same model as `model/train.py`, step by step, with
GPU support on Google Colab. It classifies wheat leaf images into 5 classes:
**healthy, leaf rust, stripe rust, powdery mildew, septoria**.

**Steps:**
1. Download / mount a public wheat-leaf dataset
2. Split into train / validation / test sets
3. Train with MobileNetV2 transfer learning (two phases)
4. Evaluate on the held-out test set
5. Download the artifacts for the FastAPI backend

In [ ]:
# 0. Check GPU (Runtime -> Change runtime type -> GPU recommended)
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

In [ ]:
# 1a. Option A: dataset on Google Drive (image-folder format)
#     /content/drive/MyDrive/wheat_data/{healthy,leaf_rust,stripe_rust,powdery_mildew,septoria}/...
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/wheat_data'

In [ ]:
# 1b. Option B: dataset from Kaggle (upload your kaggle.json first)
# Example public datasets: search kaggle.com for 'wheat leaf disease dataset'.
# !pip install -q kaggle
# from google.colab import files
# files.upload()  # select kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d <owner>/<wheat-disease-dataset> -p /content/data
# !unzip -q /content/data/*.zip -d /content/data/raw
# DATA_DIR = '/content/data/raw'

In [ ]:
# 2. Clone the repository and train using the shared training module
!git clone https://github.com/razamuzzamil07-prog/wheat-disease-detection.git
%cd wheat-disease-detection
!pip install -q -r requirements.txt

In [ ]:
# 3. Train (adjust epochs for your dataset size)
# DATA_DIR should point at the folder with one subfolder per class.
import os
os.environ['DATA_DIR'] = DATA_DIR if 'DATA_DIR' in dir() else 'data/raw'
!python model/train.py --data_dir $DATA_DIR --output_dir model/artifacts --epochs 20 --fine_tune_epochs 10

In [ ]:
# 4. Inspect evaluation outputs
from IPython.display import Image as IPyImage, display
print(open('model/artifacts/classification_report.txt').read())
display(IPyImage('model/artifacts/confusion_matrix.png'))
display(IPyImage('model/artifacts/training_history.png'))

In [ ]:
# 5. Try a prediction on one test image
import glob, json
from model.predict import predict
sample = sorted(glob.glob(os.environ['DATA_DIR'] + '/*/*.jp*g'))[0]
print(sample)
print(json.dumps(predict(sample), indent=2))

In [ ]:
# 6. Download artifacts to your machine (copy them into model/artifacts/ locally)
from google.colab import files
for f in ['wheat_disease_model.keras', 'class_indices.json',
          'classification_report.txt', 'confusion_matrix.png', 'training_history.png']:
    files.download(f'model/artifacts/{f}')